# Squid Game'deki Cam Köprüyü Python ile Simüle Etmek

[haticeozbolat.com](https://haticeozbolat.com) **Popüler Kültürü Kodluyorum** serisinin not defteri. Yazının tamamı: [YAZI LİNKİ]

Soru: 18 adımlık cam köprüde 16 oyuncu sırayla geçiyor. Her adımda iki camdan biri sağlam, biri kırılıyor. Sıranı seçebilseydin kaçıncı olmak isterdin?

Model varsayımları: her adımda doğru camı seçme şansı %50, oyuncular camları ayırt edemiyor ve öndekilerin açtığı yolu hatasız hatırlıyor. Süre sınırı ve dizideki diğer olaylar modelde yok.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

## 1. Tek bir oyunu simüle eden fonksiyon

In [ ]:
def kopruyu_gec(oyuncu=16, adim=18, rng=None, dogru_olasiligi=0.5):
    """Bir oyunu simüle eder; hayatta kalan oyuncuların sıra numaralarını döndürür."""
    rng = rng or np.random.default_rng()
    bilinen = 0                       # öndekilerin açtığı yol buraya kadar güvenli
    kalanlar = []
    for sira in range(1, oyuncu + 1):
        konum = bilinen               # bilinen yolu güvenle yürü
        hayatta = True
        while konum < adim:
            konum += 1                # bir sonraki bilinmeyen adıma bas
            if rng.random() >= dogru_olasiligi:   # yanlış cam
                hayatta = False       # düştü, ama bu adım artık biliniyor
                break
        bilinen = konum
        if hayatta:
            # bu oyuncu köprüyü bitirdi; arkadakiler de açılan yoldan geçer
            kalanlar = list(range(sira, oyuncu + 1))
            break
    return kalanlar

print("Örnek bir oyunda hayatta kalanlar:", kopruyu_gec(rng=np.random.default_rng(7)))

## 2. 100.000 kez oynatmak (birkaç saniye sürer)

In [ ]:
rng = np.random.default_rng(42)        # aynı sonucu tekrar alabilmek için tohum
N = 100_000
hayatta_kalma = np.zeros(17)           # her sıra numarası için kaç kez hayatta kaldı
kalan_sayilari = []

for _ in range(N):
    kalanlar = kopruyu_gec(rng=rng)
    kalan_sayilari.append(len(kalanlar))
    for sira in kalanlar:
        hayatta_kalma[sira] += 1

olasilik = hayatta_kalma[1:] / N
kalan_sayilari = np.array(kalan_sayilari)
print("Ortalama hayatta kalan:", kalan_sayilari.mean())
print("En az bir kişinin geçtiği oyunların oranı:", (kalan_sayilari > 0).mean())

## 3. Aynı sonucu formülle bulmak (binom dağılımı)

In [ ]:
def tam_olasilik(oyuncu=16, adim=18):
    """k. oyuncunun hayatta kalma olasılığı: 18 adımda k'den az düşüş olması."""
    return [sum(comb(adim, d) for d in range(k)) / 2 ** adim for k in range(1, oyuncu + 1)]

tam = tam_olasilik()
print(f"{'Sıra':>4}  {'Simülasyon':>10}  {'Formül':>8}")
for sira, (s, t) in enumerate(zip(olasilik, tam), start=1):
    print(f"{sira:>4}  {s:>10.1%}  {t:>8.1%}")
print(f"Beklenen hayatta kalan (formül): {sum(tam):.2f}")

## 4. Grafik

In [ ]:
MURDUM, SEFTALI, BOGURTLEN = "#2B1036", "#F6B17A", "#C2185B"
siralar = np.arange(1, 17)
renkler = [BOGURTLEN if t >= 0.5 and (i == 0 or tam[i - 1] < 0.5) else SEFTALI for i, t in enumerate(tam)]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(siralar, np.array(tam) * 100, color=renkler)
ax.axhline(50, color=MURDUM, ls="--", lw=1)
ax.set_xticks(siralar)
ax.set_xlabel("Köprüye çıkış sırası"); ax.set_ylabel("Hayatta kalma olasılığı (%)")
ax.set_title("Hayatta kalma şansı 10. sırada %50'yi geçiyor", color=MURDUM)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 5. Dizide sadece 3 kişi kaldı: bu ne kadar olası?

In [ ]:
print("Simülasyonda 3 ya da daha az kişinin kaldığı oyunlar:", f"{(kalan_sayilari <= 3).mean():.1%}")

## Kendin dene

- Köprü 30 adım olsaydı ne olurdu? `kopruyu_gec(adim=30)`
- Oyuncular camları biraz ayırt edebilseydi: `kopruyu_gec(dogru_olasiligi=0.6)`
- Hayatta kalan sayısının dağılımını histogram olarak çiz: `plt.hist(kalan_sayilari, bins=range(18))`

In [ ]:
# Denemelerini buraya yaz
